In [25]:
import sys
from pathlib import Path
sys.path.append(str(Path("..").resolve()))
import pickle 
import tqdm
import json 
import copy
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
import torch
import numpy as np
from matplotlib import colors
import torchvision
import torch.nn.functional as F
import pandas as pd
import pprint
import wandb


In [26]:

path =  '../data/results/input_combinations.pkl'



objects = []
with (open(path, "rb")) as openfile:
    while True:
        try:
            objects.append(pickle.load(openfile))
        except EOFError:
            break

In [27]:
metrics = objects[-1]


In [28]:

grouped = {}   # {inputs_key: {metric_name: [values]} }

for run_id, run_data in metrics.items():

    run = api.run(f"janpisl/proj_3/{run_id}")
    inputs = run.config.get("inputs")

    # make inputs usable as dict key
    if isinstance(inputs, list):
        inputs_key = tuple(inputs)
    else:
        inputs_key = inputs

    if inputs_key not in grouped:
        grouped[inputs_key] = {}

    m = run_data["val"]["2012_2017"]

    for k, v in m.items():
        if torch.is_tensor(v):
            v = v.item()
        if not isinstance(v, (int, float)):
            continue

        grouped[inputs_key].setdefault(k, []).append(v)

stats = {}   # {inputs_key: {metric: (mean, std)}}

for inputs_key, metric_dict in grouped.items():
    stats[inputs_key] = {}
    for metric, values in metric_dict.items():
        if len(values) >= 1:
            mean = np.mean(values)
            std = np.std(values, ddof=1) if len(values) > 1 else 0.0
            stats[inputs_key][metric] = (mean, std)

for inputs_key, metric_stats in stats.items():
    print(f"\n=== inputs = {inputs_key} ===")
    for metric, (mean, std) in metric_stats.items():
        print(f"{metric}: mean={mean:.3f}, std={std:.3f}")



=== inputs = ('deforestation_map', 'elevation', 'lulc', 'distance', 'protected') ===
val_iou: mean=0.308, std=0.001
val_f1: mean=0.470, std=0.002
val_precision: mean=0.446, std=0.004
val_recall: mean=0.497, std=0.009
val_average_precision: mean=0.458, std=0.002
val_ROC_AUC: mean=0.915, std=0.001
val_MAE: mean=34.782, std=0.678
val_MSE: mean=7877.573, std=455.740
val_PearsonR: mean=0.627, std=0.003
val_R2: mean=0.300, std=0.041
val_RMSE: mean=88.731, std=2.577
val_loss: mean=0.426, std=0.001

=== inputs = ('deforestation_map', 'protected') ===
val_iou: mean=0.272, std=0.003
val_f1: mean=0.428, std=0.004
val_precision: mean=0.404, std=0.019
val_recall: mean=0.457, std=0.017
val_average_precision: mean=0.394, std=0.003
val_ROC_AUC: mean=0.865, std=0.002
val_MAE: mean=38.334, std=2.535
val_MSE: mean=9192.561, std=959.082
val_PearsonR: mean=0.551, std=0.006
val_R2: mean=0.183, std=0.085
val_RMSE: mean=95.792, std=4.963
val_loss: mean=0.469, std=0.001

=== inputs = ('deforestation_map', 'di

In [ ]:
=== inputs = 50 ===
val_iou: mean=0.314, std=0.000
val_f1: mean=0.478, std=0.000
val_precision: mean=0.468, std=0.000
val_recall: mean=0.488, std=0.000
val_average_precision: mean=0.465, std=0.000
val_ROC_AUC: mean=0.913, std=0.000
val_MAE: mean=32.500, std=0.000
val_MSE: mean=6996.138, std=0.000
val_PearsonR: mean=0.661, std=0.000
val_R2: mean=0.378, std=0.000
val_RMSE: mean=83.643, std=0.000
val_loss: mean=0.424, std=0.000


inputs                                ///      f1                 ///      AP
('deforestation_map',)                        0.430, std=0.002      0.407, std=0.003
('deforestation_map', 'lulc')                 0.442, std=0.001      0.415, std=0.000
('deforestation_map', 'elevation')            0.464, std=0.003      0.445, std=0.002
('deforestation_map', 'elevation', 'lulc')    0.472, std=0.006      0.459, std=0.006



DOESNT WORK
('deforestation_map', 'distance')             0.414, std=0.009      0.398, std=0.003  
('deforestation_map', 'protected')            0.428, std=0.004      0.394, std=0.003
('def_map', 'elev', 'lulc', 'distance', 'protected') 0.470, std=0.002   0.458, std=0.002
